# Assignment 17: Text Cleaning, Preprocessing & NLP Pipeline
### Author: Parth Dadhaniya

**Dataset:** Self-curated dataset of 25 text samples covering movie reviews, product reviews, tweets, and support tickets (`raw_text_data.csv`).

## PART 1 — NLP Pipeline & Basic Text Cleaning
### Task 1: Understanding Raw Text Data

In [ ]:
import pandas as pd

# load dataset
df = pd.read_csv('raw_text_data.csv')
print('Dataset Shape:', df.shape)
print('\nFirst 5 Samples and Their Lengths:')
for i, row in df.head(5).iterrows():
    print(f'Sample {row["id"]} [{row["category"]}] (Length: {len(row["raw_text"])} chars):')
    print(' ', row['raw_text'])

print('\nCommon Raw Text Issues Found:')
print('- Uppercase / lowercase mismatch: ALL-CAPS words like BEST MOVIE, ASAP')
print('- Punctuation: repeated marks like !!!, ???, ...')
print('- Numbers: dates, prices, ratings like 2024, $49.99, 5/5')
print('- Extra spaces: irregular spacing around punctuation and tags')
print('- Web noise: URLs and email addresses')
print('- HTML tags: <div>, <b>, <p>, <span>')
print('- Emojis: 🔥, 🎬, 😡, 👎, ⭐')
print('- Slang and repeated letters: soooo, u, gr8, pls')

### Task 2: Basic Text Cleaning

In [ ]:
import re

def basic_clean(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)       # remove punctuation
    text = re.sub(r'\d+', '', text)           # remove numbers
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text

df['clean_text_basic'] = df['raw_text'].apply(basic_clean)

print('Original vs Cleaned (First 3 Samples):')
for i in range(3):
    print(f'\nSample {i+1} Original: ', df['raw_text'].iloc[i])
    print(f'Sample {i+1} Cleaned:  ', df['clean_text_basic'].iloc[i])

## PART 2 — Advanced Text Cleaning
### Task 3: Removing Noise

In [ ]:
def advanced_noise_removal(text):
    text = re.sub(r'<.*?>', '', text)                                           # remove html tags
    text = re.sub(r'&\w+;', '', text)                                           # remove html entities
    text = re.sub(r'https?://\S+|www\.\S+', '', text)                           # remove urls
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)  # remove emails
    text = text.encode('ascii', 'ignore').decode('ascii')                       # remove emojis
    text = re.sub(r'[^\w\s]', '', text)                                         # remove punctuation
    text = re.sub(r'\d+', '', text)                                             # remove digits
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()                                    # remove extra spaces
    return text

df['clean_text_advanced'] = df['raw_text'].apply(advanced_noise_removal)

print('Advanced Cleaning Samples:')
for i in range(3):
    print(f'\nSample {i+1} Cleaned: ', df['clean_text_advanced'].iloc[i])

### Task 4: Handling Stopwords

In [ ]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print('Total English stopwords:', len(stop_words))

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['text_no_stopwords'] = df['clean_text_advanced'].apply(remove_stopwords)

print('\nStopword Removal Samples:')
for i in range(3):
    before = df['clean_text_advanced'].iloc[i]
    after = df['text_no_stopwords'].iloc[i]
    print(f'Sample {i+1} (Words: {len(before.split())} -> {len(after.split())}):', after)

### Task 5: Handling Repeated Characters & Slang

In [ ]:
slang_dict = {
    'u': 'you',
    'gr8': 'great',
    'pls': 'please',
    'thx': 'thanks',
    'omg': 'oh my god',
    'bcoz': 'because',
    'idk': 'i do not know'
}

def clean_slang_and_repeats(text):
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    words = text.split()
    words = [slang_dict.get(w.lower(), w) for w in words]
    return ' '.join(words)

example = 'omg this phone battery is terribleeee... pls fix it u guys promised gr8 performance'
print('Original:  ', example)
print('Normalized:', clean_slang_and_repeats(example))

## PART 3 — Basic Text Preprocessing in NLP
### Task 6: Tokenization (Word and Sentence)

In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize

for sample_id in [2, 4, 14]:
    raw = df[df['id'] == sample_id]['raw_text'].iloc[0]
    sentences = sent_tokenize(raw)
    words = word_tokenize(raw)
    print(f'\nSample {sample_id}:')
    print('Raw text: ', raw)
    print('Sentences:', sentences)
    print('Words:    ', words[:8], '...')

### Task 7: Stemming

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
words = ['running', 'watched', 'acting', 'studies', 'leaves', 'caring', 'carefully', 'amaziiing']

for w in words:
    print(f'{w:<15} -> {stemmer.stem(w)}')

### Task 8: Lemmatization vs Stemming

In [ ]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
test_words = [
    ('running', 'v'),
    ('watched', 'v'),
    ('studies', 'n'),
    ('leaves', 'n'),
    ('better', 'a'),
    ('corpora', 'n'),
    ('caring', 'v'),
    ('happier', 'a')
]

for word, pos in test_words:
    stemmed = stemmer.stem(word)
    lemma = lemmatizer.lemmatize(word, pos=pos)
    print(f'{word:<12} | Stem: {stemmed:<10} | Lemma: {lemma}')

### Task 9: Final NLP Pipeline Creation

In [ ]:
def nlp_preprocess(text):
    if not isinstance(text, str):
        return ''
    # remove html, links, emails, and emojis
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'&\w+;', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    # fix repeated letters and slang
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = text.lower().strip()
    words = [slang_dict.get(w, w) for w in text.split()]
    text = ' '.join(words)
    # remove punctuation and numbers
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    # remove stopwords and lemmatize
    tokens = text.split()
    clean_tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words and len(w) > 1]
    return ' '.join(clean_tokens)

df['final_clean_text'] = df['raw_text'].apply(nlp_preprocess)

print('Final Cleaned Samples:')
for i in range(3):
    print(f'\nOriginal: {df["raw_text"].iloc[i]}')
    print(f'Cleaned:  {df["final_clean_text"].iloc[i]}')

df.to_csv('cleaned_text_final.csv', index=False)
print('\nSaved final output to cleaned_text_final.csv')

### Task 10: Observations & Insights

1. **Difference between basic and advanced cleaning:**
   - Basic cleaning only strips punctuation and numbers; on web text, it breaks URLs and HTML tags into leftover fragments.
   - Advanced cleaning targets complete URL structures, email formats, and HTML markup before word-level operations, keeping the vocabulary clean.

2. **Why lemmatization is preferred over stemming:**
   - Stemming blindly chops suffixes using heuristic rules and often creates non-words (`studies` $\to$ `studi`).
   - Lemmatization uses grammatical parts of speech and dictionary lookups to find the true root word (`studies` $\to$ `study`), preserving meaning in language models.

3. **Importance of preprocessing in NLP models:**
   - Reduces vocabulary size and dimensions, improving training speed and reducing memory use.
   - Removes noisy tokens like links and emails that do not generalize.
   - Normalizes different forms of the same word so models can detect patterns across inflections.